In [2]:
import findspark
findspark.init()

In [154]:
from pyspark.sql import SparkSession

spark = SparkSession.builder. \
    appName("pyspark-1"). \
    getOrCreate()

### Read data

In [155]:
df = spark.read.csv("/dataset/nyc-jobs.csv", header=True,
    inferSchema=True,
    multiLine=True,
    escape='\"',
    quote='\"',
    ignoreLeadingWhiteSpace=True,
    ignoreTrailingWhiteSpace=True)
df.printSchema()

root
 |-- Job ID: integer (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: integer (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: double (nullable = true)
 |-- Salary Range To: double (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locat

In [156]:
def clean_and_preprocess(df: DataFrame) -> DataFrame:
    # Step 1: Filter out rows with null Job Category
    df_cleaned = df.filter(col("Job Category").isNotNull())

    # Step 2: Estimate average salary from range
    df_with_avg = df_cleaned.withColumn(
        "Estimated Salary",
        (col("Salary Range From").cast("double") + col("Salary Range To").cast("double")) / 2
    )

    # Step 3: Convert hourly salary to annual salary
    df_final = df_with_avg.withColumn(
        "Annual Salary",
        when(col("Salary Frequency") == "Hourly", col("Estimated Salary") * 2080)
        .otherwise(col("Estimated Salary"))
    )

    return df_final
df1 = clean_and_preprocess(df)

In [157]:
#1. Whats the number of jobs posting per category (Top 10)?
def top_job_categories(df, top_n=10):
    """
    Returns the top N job categories with the highest number of job postings.

    Parameters:
    - df (DataFrame): The input DataFrame (should contain 'Job Category' and 'Job ID')
    - top_n (int): Number of top categories to return (default is 10)

    Returns:
    - DataFrame: A DataFrame of top job categories and their job counts
    """
    return (
        df.groupBy("Job Category")
          .agg(count("Job ID").alias("Number of Jobs"))
          .orderBy(desc("Number of Jobs"))
          .limit(top_n)
    )

In [158]:
result_df = top_job_categories(df1, top_n=10)
result_df.show()

+--------------------+--------------+
|        Job Category|Number of Jobs|
+--------------------+--------------+
|Engineering, Arch...|           504|
|Technology, Data ...|           313|
|       Legal Affairs|           226|
|Public Safety, In...|           182|
|Building Operatio...|           181|
|Finance, Accounti...|           169|
|Administration & ...|           134|
|Constituent Servi...|           129|
|              Health|           125|
|Policy, Research ...|           124|
+--------------------+--------------+



In [ ]:
#2. Whats the salary distribution per job category?

In [159]:
from pyspark.sql.functions import col, when, avg, min, max, count

def salary_stats_by_category(df):
    """
    Calculates salary statistics by Job Category:
    - Estimates salary from ranges
    - Converts hourly salaries to annual
    - Groups and returns count, average, min, max salary

    Parameters:
    - df (DataFrame): Input DataFrame with salary columns and Job Category

    Returns:
    - DataFrame: Aggregated salary statistics by Job Category
    """

    return df_converted.groupBy("Job Category").agg(
        count("*").alias("Job Count"),
        avg("Annual Salary").alias("Avg Salary"),
        min("Annual Salary").alias("Min Salary"),
        max("Annual Salary").alias("Max Salary")
    ).orderBy(col("Avg Salary").desc())

In [160]:
stats_df = salary_stats_by_category(df1)
stats_df.show()

+--------------------+---------+------------------+------------------+------------------+
|        Job Category|Job Count|        Avg Salary|        Min Salary|        Max Salary|
+--------------------+---------+------------------+------------------+------------------+
|Administration & ...|        2|          218587.0|          218587.0|          218587.0|
|Engineering, Arch...|        2|          198518.0|          198518.0|          198518.0|
|Engineering, Arch...|        4|          196042.5|          182500.0|          209585.0|
|Health Policy, Re...|        4|          128694.5|           94889.0|          162500.0|
|Engineering, Arch...|        2|          128247.5|          128247.5|          128247.5|
|Engineering, Arch...|        2|          128247.5|          128247.5|          128247.5|
|Communications & ...|        2|          125000.0|          125000.0|          125000.0|
|Constituent Servi...|        2|122182.31999999999|122182.31999999999|122182.31999999999|
|Administr

In [48]:
#3. Whats the job posting having the highest salary per agency?

In [161]:
from pyspark.sql.functions import col, when, dense_rank, desc
from pyspark.sql.window import Window

def highest_paid_jobs_per_agency(df):
    """
    Finds the job posting with the highest annual salary for each agency.

    Parameters:
    - df (DataFrame): Input DataFrame with salary columns and agency/job info

    Returns:
    - DataFrame: Job with the highest salary per agency
    """

    # Create window partitioned by Agency and ordered by highest salary
    window = Window.partitionBy("Agency").orderBy(desc("Annual Salary"))

    # Rank and filter to top-paid job(s) per agency
    ranked_df = df_converted.withColumn("rank", dense_rank().over(window))

    return (
        ranked_df
        .filter(col("rank") == 1)
        .select("Agency", "Job ID", "Business Title", "Annual Salary")
        .orderBy(desc("Annual Salary"))
    )

In [162]:
top_paid_jobs_df = highest_paid_jobs_per_agency(df1)
top_paid_jobs_df.show(truncate=False)

+------------------------------+------+----------------------------------------------------+-------------+
|Agency                        |Job ID|Business Title                                      |Annual Salary|
+------------------------------+------+----------------------------------------------------+-------------+
|DEPT OF ENVIRONMENT PROTECTION|396521|Deputy Commissioner, Bureau of Customer Services    |218587.0     |
|DEPT OF ENVIRONMENT PROTECTION|396521|Deputy Commissioner, Bureau of Customer Services    |218587.0     |
|POLICE DEPARTMENT             |415583|Deputy Commissioner, Public Information, M-VII      |217201.0     |
|POLICE DEPARTMENT             |415583|Deputy Commissioner, Public Information, M-VII      |217201.0     |
|DISTRICT ATTORNEY KINGS COUNTY|425494|Co-Chief Information Officer                        |191913.0     |
|DISTRICT ATTORNEY KINGS COUNTY|425494|Co-Chief Information Officer                        |191913.0     |
|NYC HOUSING AUTHORITY         |41654

In [30]:
#4. Whats the job positings average salary per agency for the last 2 years?

In [163]:
def avg_salary_per_agency_last_2_years(df):
    """
    Calculates the average annual salary per agency for job postings in the last 7 years.

    Parameters:
    - df (DataFrame): Input DataFrame with 'Posting Date', salary fields, and 'Agency'

    Returns:
    - DataFrame: Average salary per agency ordered by highest salary
    """
    # Convert Posting Date to proper date format
    df_clean = df.withColumn("Posting_Date", to_date(col("Posting Date"), "MM/dd/yyyy"))
    # Filter for last 2 years
    current_year = datetime.datetime.now().year
    df_recent = df_clean.filter(year(col("Posting_Date")) >= current_year - 7)
    # Group by agency and compute average salary
    return (
        df_salary.groupBy("Agency")
        .agg(avg("Annual Salary").alias("Avg_Salary"))
        .orderBy(col("Avg_Salary").desc())
    )

In [164]:
result_df = avg_salary_per_agency_last_2_years(df1)
result_df.show(truncate=False)

+------------------------------+-----------------+
|Agency                        |Avg_Salary       |
+------------------------------+-----------------+
|CONFLICTS OF INTEREST BOARD   |135000.0         |
|NYC EMPLOYEES RETIREMENT SYS  |98902.5          |
|FINANCIAL INFO SVCS AGENCY    |96021.12903225806|
|NYC HOUSING AUTHORITY         |88943.51428571428|
|DEPT OF DESIGN & CONSTRUCTION |87849.33802816902|
|MAYORS OFFICE OF CONTRACT SVCS|87357.14285714286|
|DEPT OF INFO TECH & TELECOMM  |85591.33177570094|
|NYC DEPT OF VETERANS' SERVICES|82653.0          |
|BUSINESS INTEGRITY COMMISSION |82117.35714285714|
|DEPARTMENT OF PROBATION       |81021.77826086956|
|OFFICE OF THE COMPTROLLER     |80960.3125       |
|HOUSING PRESERVATION & DVLPMNT|80102.37790697675|
|DEPARTMENT OF FINANCE         |79593.83333333333|
|DEPARTMENT OF CORRECTION      |78962.83239215685|
|BOARD OF CORRECTION           |78783.75         |
|LAW DEPARTMENT                |78711.02164210526|
|DEPT OF ENVIRONMENT PROTECTION

In [ ]:
#5. What are the highest paid skills in the US market?

In [165]:
from pyspark.sql.functions import col, when, avg, lower, regexp_replace, split, trim, explode
from pyspark.sql import DataFrame

def get_highest_paid_skills(df: DataFrame, top_n: int = 30) -> DataFrame:
    """
    Returns top N highest paid skills based on preferred skills and annual salary.
    
    Parameters:
        df (DataFrame): Source DataFrame with raw job data.
        top_n (int): Number of top skills to return.
    
    Returns:
        DataFrame: Top N skills with highest average salaries.
    """

    # Extract preferred skills and clean
    df_skills = df_salary.select("Preferred Skills", "Annual Salary").na.drop()

    df_skills = df_skills.withColumn("Skills_Cleaned",
        regexp_replace(lower(col("Preferred Skills")), "[^a-zA-Z0-9,./+ -]", "")
    )

    # Tokenize skills
    df_skills = df_skills.withColumn("Skill", explode(split(col("Skills_Cleaned"), "[,/;+]"))) \
                         .withColumn("Skill", trim(col("Skill")))

    # Filter empty skill strings
    df_skills = df_skills.filter(col("Skill") != "")

    # Group by skill and get average salary
    df_skill_salary = df_skills.groupBy("Skill").agg(
        avg("Annual Salary").alias("Average_Salary")
    )

    # Return top N
    return df_skill_salary.orderBy(col("Average_Salary").desc()).limit(top_n)

In [166]:
top_skills_df = get_highest_paid_skills(df1)
top_skills_df.show()

+--------------------+--------------+
|               Skill|Average_Salary|
+--------------------+--------------+
|results-driven en...|      218587.0|
|   diagnose problems|      218587.0|
|communication and...|      218587.0|
|of which at least...|      218587.0|
|continuous improv...|      218587.0|
|and implementing ...|      218587.0|
|develop and retai...|      218587.0|
|advanced degree p...|      218587.0|
|she must be an ef...|      218587.0|
|with ability to i...|      218587.0|
|capable of sustai...|      218587.0|
|the deputy commis...|      218587.0|
|goals and career ...|      218587.0|
|city and state go...|      218587.0|
|and implement act...|      218587.0|
|the following ski...|      218587.0|
|       bs degree mba|      218587.0|
|oversight counter...|      209585.0|
|values  and goals...|      209585.0|
|operations and st...|      209585.0|
+--------------------+--------------+
only showing top 20 rows



In [167]:
# Is there any correlation between the higher degree and the salary?

In [168]:
# For correlation, we convert degrees into numerical scale:

# High School → 1

# Associate's → 2

# Bachelor's → 3

# Master's → 4

# Doctorate / PhD → 5

In [169]:
from pyspark.sql.functions import col, when, lower, regexp_extract

def correlation_degree_salary(df: DataFrame) -> float:
    # Extract degree keywords
    degree_expr = lower(col("Minimum Qual Requirements"))
    df_degrees = df_salary.withColumn(
        "Degree_Level",
        when(degree_expr.contains("doctor") | degree_expr.contains("phd"), 5)
        .when(degree_expr.contains("master"), 4)
        .when(degree_expr.contains("bachelor"), 3)
        .when(degree_expr.contains("associate"), 2)
        .when(degree_expr.contains("high school"), 1)
        .otherwise(None)
    ).filter(col("Degree_Level").isNotNull() & col("Annual Salary").isNotNull())

    # 3. Compute correlation
    correlation = df_degrees.stat.corr("Degree_Level", "Annual Salary")
    return correlation

In [170]:
# This correlation value:
# Near 1 → strong positive correlation (higher degree = higher salary)
# Near 0 → no correlation
# Near -1 → inverse correlation (unlikely here)

corr_value = correlation_degree_salary(df1)
print(f"Correlation between degree and salary: {corr_value}")

Correlation between degree and salary: 0.43040703497639854


In [171]:
# after cleaning and preprocessing store the data to path
df1.write.mode("overwrite").option("header", "true").csv("/dataset/nyc-jobs_cleaned_preprocessed.csv")

### Sample function

In [48]:
def get_salary_frequency(df: DataFrame) -> list:
    row_list = df.select('Salary Frequency').distinct().collect()
    return [row['Salary Frequency'] for row in row_list]

### Example of test function

In [65]:
mock_data = [('A', 'Annual'), ('B', 'Daily')]
expected_result = ['Annual', 'Daily']

In [66]:
def test_get_salary_frequency(mock_data: list, 
                              expected_result: list,
                              schema: list = ['id', 'Salary Frequency']):  
    mock_df = spark.createDataFrame(data = mock_data, schema = schema)
    assert get_salary_frequency(mock_df) == expected_result